## Load EC3 EPD Data

### Load dataframe

In [4]:
import pandas as pd

# Load in epd_data_fly_ash_or_ggbs.csv to a dataframe
df = pd.read_csv('../02_processed_data/epd_data_fly_ash_or_ggbs.csv')


In [5]:
df.head()

,name,gwp_per_category_declared_unit,lightweight,created_on,standard_deviation,updated_on,id,open_xpd_uuid,gwp,uncertainty_factor,...,plant_or_group.id,plant_or_group.latitude,plant_or_group.type,concrete_compressive_strength_28d,cementitious.fly_ash,concrete_aggregate_size_max,cementitious.ggbs,gwp_val_per_cy,created_date_formatted,Compressive_Strength
0,Mix 57F630,355 kgCO2e,False,2023-01-15T11:30:51.091553Z,33.31495528 kgCO2e,2023-11-25T15:10:08.500691Z,eb6de10f511e4984bb04aeaa47a5f358,ec3qxf7e,355 kgCO2e,1.078980,...,1f42b9ef26994694899661fcd182e4a4,37.367974,Plant,3.93 MPa,0.25,NaN,NaN,271.417025,2023-01-15 11:30:51.091553+00:00,500
1,Mix 57F727OR,435 kgCO2e,False,2023-01-15T11:23:38.354984Z,38.17388693 kgCO2e,2023-11-25T14:44:32.247445Z,aabffeb29a1848729855d1f0381c8c50,ec3q7qjw,435 kgCO2e,1.073856,...,1f42b9ef26994694899661fcd182e4a4,37.367974,Plant,4.48 MPa,0.20,0 in,NaN,332.581425,2023-01-15 11:23:38.354984+00:00,500
2,Mix 57F727,429 kgCO2e,False,2023-01-15T11:22:25.383711Z,42.01286693 kgCO2e,2023-11-25T13:20:32.250935Z,6129df0f3d9f412b85594e981a5f9874,ec3fqzzm,429 kgCO2e,1.082420,...,1f42b9ef26994694899661fcd182e4a4,37.367974,Plant,4.48 MPa,0.20,0 in,NaN,327.994095,2023-01-15 11:22:25.383711+00:00,500
3,Mix 57F728OR,429 kgCO2e,False,2023-01-15T11:08:40.974201Z,37.64735055 kgCO2e,2023-11-24T19:59:57.493770Z,005c122c7bed45e0869959ef50052457,ec3c16jg,429 kgCO2e,1.073856,...,937cf63e18534e089c690f7a280bd32b,37.493313,Plant,4.48 MPa,0.20,5 in,NaN,327.994095,2023-01-15 11:08:40.974201+00:00,500
4,57F630,354 kgCO2e,False,2022-12-20T13:58:03.097234Z,33.22111034 kgCO2e,2023-11-25T03:35:07.071049Z,7b15d17fa1d84df086842d0fc45a74b3,ec3q1rce,354 kgCO2e,1.078980,...,937cf63e18534e089c690f7a280bd32b,37.493313,Plant,3.93 MPa,0.25,NaN,NaN,270.652470,2022-12-20 13:58:03.097234+00:00,500


In [8]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# Filter data: only include Compressive_Strength >= 2500 and values with 30+ samples
df_filtered = df[df['Compressive_Strength'] >= 2500].copy()
strength_counts = df_filtered['Compressive_Strength'].value_counts()
valid_strengths = strength_counts[strength_counts >= 30].index
df_filtered = df_filtered[df_filtered['Compressive_Strength'].isin(valid_strengths)].copy()

# Create a categorical variable for SCM type
def classify_scm(row):
    has_fly_ash = pd.notna(row.get('cementitious.fly_ash')) and row.get('cementitious.fly_ash') > 0
    has_ggbs = pd.notna(row.get('cementitious.ggbs')) and row.get('cementitious.ggbs') > 0

    if has_fly_ash and has_ggbs:
        return 'Both Fly Ash & Slag'
    elif has_fly_ash:
        return 'Fly Ash'
    elif has_ggbs:
        return 'Slag'
    else:
        return 'Neither'

df_filtered['scm_type'] = df_filtered.apply(classify_scm, axis=1)

# Round gwp values for cleaner display
df_filtered['gwp_rounded'] = df_filtered['gwp_val_per_cy'].round(0)

# Sort compressive strengths and get counts
strength_order = sorted(df_filtered['Compressive_Strength'].unique())
strength_counts_dict = df_filtered['Compressive_Strength'].value_counts().to_dict()

# Define a nice color palette for SCM types
color_map = {
    'Fly Ash': '#FF6B6B',              # Coral red
    'Slag': '#4ECDC4',                 # Turquoise
    'Both Fly Ash & Slag': '#FFE66D',  # Yellow
    #'Neither': '#95A5A6'               # Gray
}

# Create the box plot grouped by Compressive_Strength, colored by SCM type
fig = px.box(
    df_filtered,
    x='Compressive_Strength',
    y='gwp_rounded',
    title='GWP Distribution by Compressive Strength and SCM Type',
    labels={
        'Compressive_Strength': 'Compressive Strength',
        'gwp_rounded': 'GWP (kg CO₂e per cubic yard)',
        'scm_type': 'SCM Type'
    },
    category_orders={'Compressive_Strength': strength_order},
    hover_data=['scm_type', 'name'],
    color='scm_type',
    color_discrete_map=color_map
)

# Add all data points as dots with transparency and jitter
fig.update_traces(
    boxpoints='all',
    jitter=0.3,
    pointpos=0,
    marker=dict(size=5, opacity=0.6, line=dict(width=0.5, color='white'))
)

# Add vertical lines between compressive strength buckets
y_min = df_filtered['gwp_rounded'].min()
y_max = df_filtered['gwp_rounded'].max()

for i in range(len(strength_order) - 1):
    # Position line between current and next strength value
    x_position = (strength_order[i] + strength_order[i+1]) / 2

    fig.add_shape(
        type="line",
        x0=x_position,
        y0=y_min - 20,  # Extend slightly below
        x1=x_position,
        y1=y_max + 20,  # Extend slightly above
        line=dict(
            color="rgba(128, 128, 128, 0.3)",
            width=1,
            dash="dash"
        ),
        layer="below"
    )

# Update y-axis
fig.update_yaxes(
    title_text='GWP (kg CO₂e per cubic yard)',
    showgrid=True,
    gridcolor='lightgrey'
)

# Update x-axis to include counts in labels
x_labels_with_counts = [f"{int(strength)} psi [{strength_counts_dict[strength]}]"
                        for strength in strength_order]

fig.update_xaxes(
    ticktext=x_labels_with_counts,
    tickvals=strength_order,
    tickangle=-45
)

# Update layout
fig.update_layout(
    title={
        'text': 'GWP Distribution by Compressive Strength and SCM Type',
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 16, 'family': 'Arial', 'color': 'black'}
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    width=1200,
    legend={
        'title': 'SCM Type',
        'orientation': 'v',
        'yanchor': 'top',
        'y': 0.99,
        'xanchor': 'left',
        'x': 1.01
    }
)

# Show summary statistics
print(f"Total samples: {len(df_filtered)}")
print(f"\nSamples by Compressive Strength:")
for strength in strength_order:
    count = strength_counts_dict[strength]
    print(f"  {int(strength)} psi: {count} samples")
print(f"\nSamples by SCM Type:")
print(df_filtered['scm_type'].value_counts())

# Show the plot
fig.show()

Total samples: 4625

Samples by Compressive Strength:
  2500 psi: 65 samples
  3000 psi: 607 samples
  3500 psi: 328 samples
  4000 psi: 1406 samples
  4500 psi: 500 samples
  5000 psi: 799 samples
  5500 psi: 109 samples
  6000 psi: 606 samples
  7000 psi: 103 samples
  8000 psi: 102 samples

Samples by SCM Type:
scm_type
Fly Ash                3734
Slag                    809
Both Fly Ash & Slag      82
Name: count, dtype: int64


### Save Plot

In [9]:
# Save the plot as HTML
import os
output_path = '../05_outputs/gwp_by_compressive_strength_scm.html'
fig.write_html(output_path, include_plotlyjs='cdn')